# Actividad 2: regresión lineal y ANOVA
**Materia:** Aplicación de métodos multivariados en ciencia de datos  
**Nombre:** Mateo Flores



In [3]:
import pandas as pd
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

datos = pd.DataFrame({
    'Estudiante': range(1, 11),
    'Horas': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Nota': [3.2, 4.5, 5.0, 6.8, 7.0, 8.5, 9.0, 9.2, 9.8, 10.0]
})
datos

,Estudiante,Horas,Nota
0,1,1,3.2
1,2,2,4.5
2,3,3,5.0
3,4,4,6.8
4,5,5,7.0
5,6,6,8.5
6,7,7,9.0
7,8,8,9.2
8,9,9,9.8
9,10,10,10.0


## 1. Modelo lineal por MCO
MCO significa **Mínimos Cuadrados Ordinarios**. El modelo es $Nota = \beta_0 + \beta_1(Horas) + error$.

In [4]:
modelo = ols('Nota ~ Horas', data=datos).fit()
b0, b1 = modelo.params
print(f'Modelo: Nota = {b0:.4f} + {b1:.4f}(Horas)')
print(f'R² = {modelo.rsquared:.4f}')

Modelo: Nota = 3.0533 + 0.7721(Horas)
R² = 0.9502


El modelo estima que cada hora adicional de estudio aumenta la nota en aproximadamente **0.7721 puntos**.

## 2. Tabla ANOVA

In [5]:
anova = anova_lm(modelo)
anova.round(4)

,df,sum_sq,mean_sq,F,PR(>F)
Horas,1.0,49.1841,49.1841,152.7529,0.0
Residual,8.0,2.5759,0.3220,NaN,NaN


## 3. Pruebas F y t
En ambas pruebas se contrasta $H_0: \beta_1=0$ contra $H_1: \beta_1\neq0$.

In [6]:
F = float(modelo.fvalue)
p_F = float(modelo.f_pvalue)
t = modelo.tvalues['Horas']
p_t = modelo.pvalues['Horas']

print(f'Prueba F: F = {F:.4f}, p = {p_F:.8f}')
print(f'Prueba t: t = {t:.4f}, p = {p_t:.8f}')
print('Conclusión: se rechaza H0; las horas explican significativamente la nota.')

Prueba F: F = 152.7529, p = 0.00000171
Prueba t: t = 12.3593, p = 0.00000171
Conclusión: se rechaza H0; las horas explican significativamente la nota.


## 4. Relación entre F y t
En una regresión lineal con una sola variable explicativa se cumple $F=t^2$.

In [7]:
print(f'F   = {F:.4f}')
print(f't²  = {t**2:.4f}')
print('¿Son iguales?', round(F, 10) == round(t**2, 10))

F   = 152.7529
t²  = 152.7529
¿Son iguales? True


## 5. Intervalos de confianza de los coeficientes (95%)

In [8]:
ic_coeficientes = modelo.conf_int(alpha=0.05)
ic_coeficientes.columns = ['Límite inferior', 'Límite superior']
ic_coeficientes.round(4)

,Límite inferior,Límite superior
Intercept,2.1594,3.9472
Horas,0.6281,0.9162


Como el intervalo de la pendiente no contiene cero, la relación entre horas y nota es significativa.

## 6. Tabla de IC de la media e IP (95%)
El **IC** estima la nota media y el **IP** el posible valor de una nota individual.

In [9]:
pred = modelo.get_prediction(datos[['Horas']]).summary_frame(alpha=0.05)
tabla_ic_ip = pd.DataFrame({
    'Horas': datos['Horas'],
    'Nota estimada': pred['mean'],
    'IC media inferior': pred['mean_ci_lower'],
    'IC media superior': pred['mean_ci_upper'],
    'IP inferior': pred['obs_ci_lower'],
    'IP superior': pred['obs_ci_upper']
})
tabla_ic_ip.round(3)

,Horas,Nota estimada,IC media inferior,IC media superior,IP inferior,IP superior
0,1,3.825,3.056,4.595,2.308,5.343
1,2,4.598,3.945,5.250,3.136,6.060
2,3,5.370,4.821,5.918,3.951,6.789
3,4,6.142,5.675,6.609,4.753,7.531
4,5,6.914,6.494,7.334,5.540,8.288
5,6,7.686,7.266,8.106,6.312,9.060
6,7,8.458,7.991,8.925,7.069,9.847
7,8,9.230,8.682,9.779,7.811,10.649
8,9,10.002,9.350,10.655,8.540,11.464
9,10,10.775,10.005,11.544,9.257,12.292


## Conclusión
El modelo obtenido es $Nota = 3.0533 + 0.7721(Horas)$. Explica aproximadamente el **95.02%** de la variación de las notas. Como las pruebas F y t tienen un valor p menor que 0.05, existe una relación lineal significativa entre las horas de estudio y la nota.